# Kaggle Save & Run All — đánh giá QTDB/BW từ checkpoint 100 epoch

Notebook này **chỉ đánh giá**, không train lại. Trước khi chạy, dùng **Add Input** để gắn Dataset hoặc Notebook Output chứa đúng `best_protocol1.pth` và `best_protocol2.pth`. Bật **GPU** và **Internet**, sau đó chọn **Save Version → Save & Run All**.

Notebook tự tìm checkpoint trong `/kaggle/input`, tạo lại 14 QTDB test records cùng hai protocol NSTDB BW theo `rule.md`, chạy DDIM K = 1/3/5/10 và lưu kết quả vào `/kaggle/working/qtdb_rule_evaluation_100epoch/`.

## 1. Cài thư viện

In [ ]:
!pip -q install 'wfdb>=4.1,<5' 'scipy>=1.10' 'pandas>=2.0' 'matplotlib>=3.7' 'pyyaml>=6.0'


## 2. Cấu hình Save & Run All

In [ ]:
import hashlib
import json
import math
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import signal
import torch
import wfdb
from IPython.display import display

SEED = 1234
TEST_RECORDS = (
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
)
TARGET_FS = 360
LENGTH = 512
MAX_UNPADDED = 496
INSERT_OFFSET = 16
BATCH_SIZE = 50
K_VALUES = (1, 3, 5, 10)
DDIM_STEPS = 50
DDIM_ETA = 0.0
PREPROCESSING_VERSION = 'qtdb_pu1_p_and_beat_40ms_reflect100ms_360hz_endpoints_pad16_v2'
PHYSIONET_QT = 'qtdb/1.0.0'
PHYSIONET_NST = 'nstdb/1.0.0'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('/kaggle/working/qtdb_rule_evaluation_100epoch')
OUT.mkdir(parents=True, exist_ok=True)

# Chỉ sửa hai giới hạn này để smoke test; để None khi báo cáo benchmark.
MAX_TEST_RECORDS = None
MAX_TEST_SEGMENTS = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE, '| output:', OUT.resolve())


## 3. Kiến trúc HNF U-Net + FiLM + attention

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
import torch
import torch.nn as nn

class HNFBlock(nn.Module):
    """
    Half Normalized Filter block.
    Multi-scale convs → concat → conv → half‑instance norm → residual.
    """
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k,
                      padding=k//2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        # Half‑instance norm: chia channel làm 2 nửa
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        multi_out = [conv(x) for conv in self.multi_convs]
        out = torch.cat(multi_out, dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out_norm = self.half_inst_norm(out[:, :half, :])
        out = torch.cat([out_norm, out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)
import torch
import torch.nn as nn
import math

class BridgeBlock(nn.Module):
    """FiLM‑based conditioning on noise level (sqrt(alpha_bar))."""
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(
            nn.Linear(emb_dim, features * 2),
            nn.SiLU()
        )

    def forward(self, x, alpha_bar):
        # alpha_bar: (B,) or scalar
        # Ép phẳng alpha_bar về dạng 1D (Batch,) để tránh bị dư chiều -- CỰC QUAN TRỌNG
        alpha_bar = alpha_bar.view(-1)
        emb = self.sinusoidal_embedding(alpha_bar)
        scale_shift = self.film(emb)   # (B, 2*features)
        scale, shift = scale_shift.chunk(2, dim=1)
        scale = scale.unsqueeze(-1)
        shift = shift.unsqueeze(-1)
        return x * (1 + scale) + shift

    def sinusoidal_embedding(self, x):
        if x.dim() == 0:
            x = x.unsqueeze(0)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb
import torch
import torch.nn as nn

class SelfAttention1D(nn.Module):
    """1D Self‑Attention (multi‑head) dùng tại bottleneck."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels, "channels phải chia hết cho num_heads"

        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, L = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, L)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        # q, k, v: (B, num_heads, head_dim, L)
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        out = out.reshape(B, C, L)
        return self.proj(out)
import torch
import torch.nn as nn

class UNet1D(nn.Module):
    """
    1D U‑Net với 4 levels, skip connections, HNF blocks,
    Bridge FiLM (timestep), và Self‑Attention tại bottleneck.
    """
    def __init__(self, in_channels=24, base_channels=64, emb_dim=128, out_channels=12):
        """
        in_channels = 24 vì concat x_t (12 channels) và condition (12 channels)
        """
        super().__init__()
        # ---------- ENCODER ----------
        # Level 1 (input -> 64)
        self.enc1 = HNFBlock(in_channels, base_channels)
        self.bridge1 = BridgeBlock(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels*2, kernel_size=4, stride=2, padding=1)

        # Level 2 (64 -> 128)
        self.enc2 = HNFBlock(base_channels*2, base_channels*2)
        self.bridge2 = BridgeBlock(base_channels*2, emb_dim)
        self.down2 = nn.Conv1d(base_channels*2, base_channels*4, kernel_size=4, stride=2, padding=1)

        # Level 3 (128 -> 256)
        self.enc3 = HNFBlock(base_channels*4, base_channels*4)
        self.bridge3 = BridgeBlock(base_channels*4, emb_dim)
        self.down3 = nn.Conv1d(base_channels*4, base_channels*8, kernel_size=4, stride=2, padding=1)

        # Level 4 (256 -> 512) – bottleneck
        self.enc4 = HNFBlock(base_channels*8, base_channels*8)
        self.bridge4 = BridgeBlock(base_channels*8, emb_dim)
        # Self‑Attention tại bottleneck
        self.attn = SelfAttention1D(base_channels*8)

        # ---------- DECODER ----------
        # Level 4 -> 3
        self.up4 = nn.ConvTranspose1d(base_channels*8, base_channels*4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlock(base_channels*8, base_channels*4)   # concat với skip từ enc3

        # Level 3 -> 2
        self.up3 = nn.ConvTranspose1d(base_channels*4, base_channels*2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlock(base_channels*4, base_channels*2)   # concat với skip từ enc2

        # Level 2 -> 1
        self.up2 = nn.ConvTranspose1d(base_channels*2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlock(base_channels*2, base_channels)     # concat với skip từ enc1

        # Output
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        """
        x: latent (x_t), cond: noisy observation, noise_scale: sqrt(alpha_bar)
        """
        # Concatenate x and cond along channel dimension
        inp = torch.cat([x, cond], dim=1)   # (B, 2, L)

        # ----- Encoder -----
        e1 = self.enc1(inp)
        e1 = self.bridge1(e1, noise_scale)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        e2 = self.bridge2(e2, noise_scale)
        d2 = self.down2(e2)

        e3 = self.enc3(d2)
        e3 = self.bridge3(e3, noise_scale)
        d3 = self.down3(e3)

        e4 = self.enc4(d3)
        e4 = self.bridge4(e4, noise_scale)
        # Bottleneck self‑attention
        e4 = self.attn(e4)

        # ----- Decoder (with skip connections) -----
        u4 = self.up4(e4)
        u4 = torch.cat([u4, e3], dim=1)   # skip từ enc3
        d4 = self.dec4(u4)

        u3 = self.up3(d4)
        u3 = torch.cat([u3, e2], dim=1)   # skip từ enc2
        d3 = self.dec3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, e1], dim=1)   # skip từ enc1
        d2 = self.dec2(u2)

        out = self.final(d2)
        return out

## 4. DDPM/DDIM và loss đa miền của nghiên cứu

In [ ]:
from functools import partial
from inspect import isfunction
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from inspect import isfunction

# ----- Bắt buộc phải có torchaudio hoặc dùng torch.stft -----
# Nếu không có torchaudio, ta dùng torch.stft nguyên bản
def stft_loss(pred, target, n_fft=128, hop_length=64):
    """
    Tính MSE loss trên phổ STFT giữa pred và target.
    """
    # (B, C, L) -> (B*C, L)
    B, C, L = pred.shape
    pred = pred.view(-1, L)
    target = target.view(-1, L)
    
    # STFT: returns (B, freq_bins, time_frames, 2)
    spec_pred = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, 
                           return_complex=True)
    spec_target = torch.stft(target, n_fft=n_fft, hop_length=hop_length,
                             return_complex=True)
    
    # Lấy magnitude (độ lớn)
    mag_pred = torch.abs(spec_pred)
    mag_target = torch.abs(spec_target)
    
    return F.mse_loss(mag_pred, mag_target)


def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


class DDPM(nn.Module):
    def __init__(self, base_model, config, device, conditional=True):
        super().__init__()
        self.device = device
        self.model = base_model
        self.config = config
        self.conditional = conditional
        
        # ----------------------
        # 👉 THÊM: Loss weights
        # ----------------------
        self.lambda_time = config['train'].get('lambda_time', 1.0)
        self.lambda_freq = config['train'].get('lambda_freq', 0.1)  # cân bằng
        
        self.loss_func_l1 = nn.L1Loss(reduction='sum').to(device)
        
        config_diff = config["diffusion"]
        self.num_steps = config_diff["num_steps"]
        self.set_new_noise_schedule(config_diff, device)
        
    def make_beta_schedule(self, schedule='linear', n_timesteps=1000, start=1e-5, end=1e-2):
        if schedule == 'linear':
            betas = torch.linspace(start, end, n_timesteps)
        elif schedule == "quad":
            betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
        elif schedule == "sigmoid":
            betas = torch.linspace(-6, 6, n_timesteps)
            betas = torch.sigmoid(betas) * (end - start) + start
        return betas
    
    def set_new_noise_schedule(self, config_diff, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)
        
        betas = self.make_beta_schedule(
            schedule=config_diff["schedule"], 
            n_timesteps=config_diff["num_steps"],
            start=config_diff["beta_start"], 
            end=config_diff["beta_end"]
        )
        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas
        
        alphas = 1. - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod_prev = np.sqrt(np.append(1., alphas_cumprod))
        
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev', to_torch(alphas_cumprod_prev))
        self.register_buffer('sqrt_alphas_cumprod', to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('log_one_minus_alphas_cumprod', to_torch(np.log(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod - 1)))
        
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        self.register_buffer('posterior_variance', to_torch(posterior_variance))
        self.register_buffer('posterior_log_variance_clipped', 
                             to_torch(np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', 
                             to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', 
                             to_torch((1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))
        
    def predict_start_from_noise(self, x_t, t, noise):
        # Nắn lại chiều tensor hệ số thành (Batch, 1, 1) để nhân đúng với tensor tín hiệu 3D
        coef1 = self.sqrt_recip_alphas_cumprod[t].view(-1, 1, 1)
        coef2 = self.sqrt_recipm1_alphas_cumprod[t].view(-1, 1, 1)
        
        return coef1 * x_t - coef2 * noise
    
    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * x_start + \
                         self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]
        return posterior_mean, posterior_log_variance_clipped
    
    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):
        batch_size = x.shape[0]
        noise_level = torch.FloatTensor(
            [self.sqrt_alphas_cumprod_prev[t+1]]
        ).repeat(batch_size, 1).to(x.device)
        if condition_x is not None:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, condition_x, noise_level))
        else:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, noise_level))
        if clip_denoised:
            x_recon.clamp_(-1., 1.)
        model_mean, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance
    
    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised=False, condition_x=None):
        model_mean, model_log_variance = self.p_mean_variance(
            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()

    @torch.no_grad()
    def p_sample_loop(self, x_in, continous=False):
        device = self.betas.device
        sample_inter = (1 | (self.num_steps//10))
        if not self.conditional:
            shape = x_in
            cur_x = torch.randn(shape, device=device)
            ret_x = cur_x
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i)
                if i % sample_inter == 0:
                    ret_x = torch.cat([ret_x, cur_x], dim=0)
        else:
            x = x_in
            shape = x.shape
            cur_x = torch.randn(shape, device=device)
            ret_x = [cur_x]
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i, condition_x=x)
                if i % sample_inter == 0:
                    ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]    
    
    @torch.no_grad()
    def sample(self, batch_size=1, shape=[1, 512], continous=False):
        return self.p_sample_loop((batch_size, shape[0], shape[1]), continous)
    
    # ==========================================================
    # 👉 DDIM SAMPLER (đã có sẵn trong source gốc)
    # ==========================================================
    @torch.no_grad()
    def ddim_sample_loop(self, x_in, ddim_timesteps=50, ddim_eta=0.0, num_shots=1):
        device = self.betas.device
        T = self.num_steps
        S = ddim_timesteps
        tau = [int(np.floor((T / (S ** 2)) * (i ** 2))) for i in range(S + 1)]
        
        alphas_cumprod_with_1 = torch.cat([
            torch.tensor([1.0], dtype=self.alphas_cumprod.dtype, device=device),
            self.alphas_cumprod
        ]).float()
        
        out_accum = 0.0
        for shot in range(num_shots):
            shape = x_in.shape
            x = torch.randn(shape, device=device)
            for i in reversed(range(1, S + 1)):
                t = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.FloatTensor(
                    [self.sqrt_alphas_cumprod_prev[t]]
                ).repeat(x.shape[0], 1).to(device)
                if not self.conditional:
                    eps = self.model(x, noise_level)
                else:
                    eps = self.model(x, x_in, noise_level)
                    
                a_t = alphas_cumprod_with_1[t]
                a_t_prev = alphas_cumprod_with_1[t_prev]
                x0_pred = (x - torch.sqrt(1. - a_t) * eps) / torch.sqrt(a_t)
                if t_prev == 0:
                    sigma_t = 0.0
                else:
                    sigma_t = ddim_eta * torch.sqrt((1. - a_t_prev) / (1. - a_t)) * \
                              torch.sqrt(torch.clamp(1. - a_t / a_t_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1. - a_t_prev - sigma_t**2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_t_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots

    @torch.no_grad()
    def denoising(self, x_in, continous=False, use_ddim=False, ddim_steps=50, ddim_eta=0.0, num_shots=1):
        if use_ddim:
            return self.ddim_sample_loop(x_in, ddim_timesteps=ddim_steps, ddim_eta=ddim_eta, num_shots=num_shots)
        else:
            if num_shots > 1:
                out_accum = 0.0
                for _ in range(num_shots):
                    out_accum += self.p_sample_loop(x_in, continous)
                return out_accum / num_shots
            return self.p_sample_loop(x_in, continous)
    
    def q_sample_loop(self, x_start, continous=False):
        sample_inter = (1 | (self.num_steps//10))
        ret_x = [x_start]
        cur_x = x_start
        for t in range(1, self.num_steps+1):
            B,C,L = cur_x.shape
            continuous_sqrt_alpha_cumprod = torch.FloatTensor(
                np.random.uniform(
                    self.sqrt_alphas_cumprod_prev[t-1],
                    self.sqrt_alphas_cumprod_prev[t],
                    size=B
                )
            ).to(cur_x.device)
            continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
            noise = torch.randn_like(cur_x)
            cur_x = self.q_sample(
                x_start=cur_x, 
                continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
                noise=noise
            )
            if t % sample_inter == 0:
                ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]
    
    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):
        noise = default(noise, lambda: torch.randn_like(x_start))
        return continuous_sqrt_alpha_cumprod * x_start + \
               (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise
    
    # ==========================================================
    # 👉 HÀM LOSS CHÍNH (ĐÃ THÊM STFT)
    # ==========================================================
    def p_losses(self, x_in, y_in, noise=None):
        """
        Multi‑domain Loss:
        - L1 trên noise (miền thời gian)
        - STFT Loss trên tín hiệu dự đoán (miền tần số)
        """
        x_start = x_in
        B, C, L = x_start.shape

        # Lấy ngẫu nhiên các giá trị t khác nhau cho từng mẫu trong Batch, trực tiếp trên thiết bị (GPU)
        t = torch.randint(0, self.num_steps, (B,), device=x_start.device).long()
        # 1. Chuyển mảng NumPy sang PyTorch Tensor và đẩy lên cùng GPU với t
        sqrt_alphas_prev_tensor = torch.tensor(self.sqrt_alphas_cumprod_prev, dtype=torch.float32, device=t.device)
        # 2. Lấy giới hạn dưới và trên. 
        # (Lưu ý: Do t giờ chạy từ 0 đến num_steps-1, ta dùng chỉ số t và t+1 để thay thế cho t-1 và t nhằm tránh lỗi âm)
        lower = sqrt_alphas_prev_tensor[t]
        upper = sqrt_alphas_prev_tensor[t+1]
        # 3. Sinh số ngẫu nhiên uniform hoàn toàn bằng PyTorch trực tiếp trên GPU
        continuous_sqrt_alpha_cumprod = lower + torch.rand(B, device=t.device) * (upper - lower)
        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
        noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(
            x_start=x_start, 
            continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
            noise=noise
        )

        # Dự đoán noise
        if not self.conditional:
            x_recon = self.model(x_noisy, continuous_sqrt_alpha_cumprod)
        else:
            x_recon = self.model(x_noisy, y_in, continuous_sqrt_alpha_cumprod)

        # ---------- 1. L1 Loss (miền thời gian) ----------
        loss_l1 = self.loss_func_l1(noise, x_recon)  # (B*C*L)

        # ---------- 2. STFT Loss (miền tần số) ----------
        # Dự đoán x0 từ noise đã dự đoán
        x0_pred = self.predict_start_from_noise(x_noisy, t, x_recon)
        
        # Tính STFT loss giữa x0_pred và x_start (clean gốc)
        loss_stft = stft_loss(x0_pred, x_start, n_fft=128, hop_length=64)
        
        # ---------- 3. Tổng hợp ----------
        total_loss = self.lambda_time * loss_l1 + self.lambda_freq * loss_stft
        return total_loss
    
    def forward(self, x, y, *args, **kwargs):
        return self.p_losses(x, y, *args, **kwargs)


class EMA(object):
    def __init__(self, mu=0.999):
        self.mu = mu
        self.shadow = {}

    def register(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name].data = (1. - self.mu) * param.data + self.mu * self.shadow[name].data

    def ema(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name].data)

    def ema_copy(self, module):
        module_copy = type(module)(module.config).to(module.config.device)
        module_copy.load_state_dict(module.state_dict())
        self.ema(module_copy)
        return module_copy

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state_dict):
        self.shadow = state_dict

## 5. Tự tìm và nạp hai checkpoint best

In [ ]:
# Gắn Kaggle Dataset/Notebook Output chứa đúng hai checkpoint bằng Add Input.
CHECKPOINT_FILENAMES = {
    1: 'best_protocol1.pth',
    2: 'best_protocol2.pth',
}
DECLARED_TRAIN_EPOCHS = 100
OUT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'train': {'feats': 80, 'batch_size': 96, 'epochs': DECLARED_TRAIN_EPOCHS,
              'lr': 1e-4, 'lambda_time': 1.0, 'lambda_freq': 0.1},
    'diffusion': {'beta_start': 1e-4, 'beta_end': 0.5,
                  'num_steps': 50, 'schedule': 'quad'},
}

def locate_unique_checkpoint(filename):
    root = Path('/kaggle/input')
    matches = sorted(path for path in root.rglob(filename) if path.is_file())
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Cần đúng một file {filename} trong Kaggle Add Input; tìm thấy {len(matches)}: {matches}'
        )
    return matches[0]

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

MODELS, CHECKPOINT_INFO = {}, []
for protocol, filename in CHECKPOINT_FILENAMES.items():
    path = locate_unique_checkpoint(filename)
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(payload, dict) or 'state_dict' not in payload:
        raise RuntimeError(f'{filename} không phải checkpoint best do notebook train tạo ra')
    if int(payload.get('protocol', -1)) != protocol:
        raise RuntimeError(f'{filename}: protocol trong checkpoint không khớp')
    saved_diffusion = payload.get('config', {}).get('diffusion', {})
    if saved_diffusion and saved_diffusion != CONFIG['diffusion']:
        raise RuntimeError(f'{filename}: cấu hình diffusion không khớp: {saved_diffusion}')
    model = DDPM(UNet1D(2, 80, 128, 1), CONFIG, DEVICE).to(DEVICE)
    model.load_state_dict(payload['state_dict'], strict=True)
    model.eval()
    probe = torch.zeros(1, 1, LENGTH, device=DEVICE)
    with torch.inference_mode():
        probe_output = model.model(probe, probe, torch.ones(1, 1, device=DEVICE))
    assert probe_output.shape == probe.shape and torch.isfinite(probe_output).all()
    MODELS[protocol] = model
    info = {
        'protocol': protocol, 'path': str(path), 'sha256': sha256(path),
        'parameter_count': sum(p.numel() for p in model.parameters()),
        'selected_epoch': int(payload.get('epoch', -1)) + 1,
        'best_validation_loss': float(payload.get('best_validation_loss', np.nan)),
        'declared_training_completed_epochs': DECLARED_TRAIN_EPOCHS,
        'saved_training_config': payload.get('config', {}),
    }
    CHECKPOINT_INFO.append(info)
    print(f"Protocol {protocol}: strict load OK; best epoch={info['selected_epoch']}; "
          f"validation={info['best_validation_loss']:.6f}; {path}")
display(pd.DataFrame(CHECKPOINT_INFO))


## 6. Hàm tạo QTDB ground truth

In [ ]:
BEAT_SYMBOLS = set('NLRBAaJSVrFejnE/fQ?')

def p_onsets(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample, dtype=np.int64)
    return np.asarray([int(samples[i]) for i, symbol in enumerate(symbols)
                       if symbol == '(' and 'p' in symbols[i + 1:i + 4]], dtype=np.int64)

def r_locations(annotation):
    return np.asarray([int(sample) for sample, symbol in zip(annotation.sample, annotation.symbol)
                       if symbol in BEAT_SYMBOLS], dtype=np.int64)

def resample_reflect(segment, original_fs):
    pad = min(max(1, int(round(0.1 * original_fs))), len(segment) - 1)
    padded = np.pad(segment, (pad, pad), mode='reflect')
    new_length = int(round(len(padded) * TARGET_FS / original_fs))
    resampled = signal.resample(padded, new_length).astype(np.float32)
    target_pad = int(round(pad * TARGET_FS / original_fs))
    expected = int(round(len(segment) * TARGET_FS / original_fs))
    return resampled[target_pad:target_pad + expected]

def clean_segments_for_record(name):
    record = wfdb.rdrecord(name, pn_dir=PHYSIONET_QT, channels=[0])
    # QTDB does not provide .atr for every record (e.g. sel30).
    # .pu1 provides P boundaries and beat fiducials throughout each record.
    pu = wfdb.rdann(name, 'pu1', pn_dir=PHYSIONET_QT)
    fs = float(record.fs)
    ecg = record.p_signal[:, 0].astype(np.float32)
    starts = p_onsets(pu) - int(round(0.04 * fs))
    starts = starts[(starts >= 0) & (starts < len(ecg))]
    peaks = r_locations(pu)
    beats, rows = [], []
    audit = {'record_id': name, 'p_onsets': len(starts),
             'candidate_intervals': max(0, len(starts) - 1),
             'rejected_short_interval': 0, 'rejected_multiple_r': 0,
             'rejected_length': 0, 'rejected_nonfinite_or_flat': 0}
    resampled_lengths = []
    for segment_index, (start, end) in enumerate(zip(starts[:-1], starts[1:])):
        if end <= start + 2:
            audit['rejected_short_interval'] += 1
            continue
        r_count = int(np.count_nonzero((peaks >= start) & (peaks < end)))
        if r_count > 1:
            audit['rejected_multiple_r'] += 1
            continue
        segment = resample_reflect(ecg[start:end], fs)
        resampled_lengths.append(len(segment))
        if not 2 <= len(segment) <= MAX_UNPADDED:
            audit['rejected_length'] += 1
            continue
        segment = segment - 0.5 * (segment[0] + segment[-1])
        clean = np.zeros(LENGTH, dtype=np.float32)
        clean[INSERT_OFFSET:INSERT_OFFSET + len(segment)] = segment
        if not np.all(np.isfinite(clean)) or np.ptp(clean) <= 0:
            audit['rejected_nonfinite_or_flat'] += 1
            continue
        beats.append(clean)
        rows.append({'sample_id': f'{name}:{segment_index}', 'patient_id': name,
                     'record_id': name, 'clean_segment_id': f'{name}:{segment_index}',
                     'start_sample': int(start), 'end_sample': int(end),
                     'unpadded_length': len(segment), 'r_count': r_count,
                     'annotation_source': 'pu1'})
    audit['min_resampled_length'] = min(resampled_lengths) if resampled_lengths else np.nan
    audit['max_resampled_length'] = max(resampled_lengths) if resampled_lengths else np.nan
    audit['accepted_segments'] = len(beats)
    return beats, rows, audit



## 7. Tải NSTDB BW

In [ ]:
noise_record = wfdb.rdrecord('bw', pn_dir=PHYSIONET_NST)
BW = noise_record.p_signal.astype(np.float32)
if int(noise_record.fs) != TARGET_FS:
    BW = signal.resample(BW, int(round(len(BW) * TARGET_FS / float(noise_record.fs))), axis=0).astype(np.float32)
assert BW.ndim == 2 and BW.shape[1] >= 2 and np.isfinite(BW).all()
MIDPOINT = len(BW) // 2
print('NSTDB BW:', BW.shape, '| midpoint:', MIDPOINT)


## 8. Mở 14 QTDB test records

In [ ]:
selected_records = TEST_RECORDS[:MAX_TEST_RECORDS] if MAX_TEST_RECORDS else TEST_RECORDS
all_beats, clean_rows, test_audit_rows = [], [], []
for record_name in tqdm(selected_records, desc='QTDB test records'):
    beats, rows, audit = clean_segments_for_record(record_name)
    test_audit_rows.append(audit)
    all_beats.extend(beats)
    clean_rows.extend(rows)
    print(record_name, len(beats))
    if MAX_TEST_SEGMENTS and len(all_beats) >= MAX_TEST_SEGMENTS:
        all_beats = all_beats[:MAX_TEST_SEGMENTS]
        clean_rows = clean_rows[:MAX_TEST_SEGMENTS]
        break
pd.DataFrame(test_audit_rows).to_csv(OUT / 'test_record_audit.csv', index=False)
if not all_beats:
    raise RuntimeError('No valid QTDB test segments after rule.md filtering; inspect test_record_audit.csv')
CLEAN = np.stack(all_beats).astype(np.float32)
CLEAN_META = pd.DataFrame(clean_rows)
assert CLEAN.shape == (len(CLEAN_META), LENGTH)
assert CLEAN_META.sample_id.is_unique
assert set(CLEAN_META.record_id).issubset(TEST_RECORDS)
np.save(OUT / 'clean_test.npy', CLEAN)
print('Clean test segments:', len(CLEAN), '| shape:', CLEAN.shape)


## 9. Tạo hai protocol BW và manifest

In [ ]:
def make_protocol_pairs(protocol):
    channel = 1 if protocol == 1 else 0  # Zero-based: P1 tests channel 2; P2 tests channel 1.
    noise = BW[MIDPOINT:, channel]
    patches = len(noise) // LENGTH
    assert patches > 0
    rng = np.random.RandomState(SEED)
    r_values = rng.randint(20, 200, size=len(CLEAN)) / 100.0
    noisy = np.empty_like(CLEAN)
    rows = []
    for index, (clean, r) in enumerate(zip(CLEAN, r_values)):
        relative_start = (index % patches) * LENGTH
        patch = noise[relative_start:relative_start + LENGTH]
        clean_range, noise_range = np.ptp(clean), np.ptp(patch)
        if clean_range <= 0 or noise_range <= 0:
            raise ValueError(f'Zero amplitude range at sample {index}, protocol {protocol}')
        alpha = float(r) * float(clean_range) / float(noise_range)
        noisy[index] = clean + alpha * patch
        row = CLEAN_META.iloc[index].to_dict()
        row.update({'protocol': protocol, 'split': 'test', 'noise_type': 'BW',
                    'noise_source_id': 'nstdb/1.0.0/bw:second_half',
                    'noise_channel': channel + 1,
                    'noise_start': MIDPOINT + relative_start,
                    'r': float(r), 'sampling_rate': TARGET_FS,
                    'preprocessing_version': PREPROCESSING_VERSION, 'seed': SEED})
        rows.append(row)
    return noisy, pd.DataFrame(rows)

NOISY, MANIFEST = {}, {}
for protocol in (1, 2):
    NOISY[protocol], MANIFEST[protocol] = make_protocol_pairs(protocol)
    np.save(OUT / f'noisy_protocol{protocol}.npy', NOISY[protocol])
manifest = pd.concat([MANIFEST[1], MANIFEST[2]], ignore_index=True)
assert len(manifest) == 2 * len(CLEAN)
assert not manifest.duplicated(['sample_id', 'protocol']).any()
assert manifest.groupby('protocol').size().eq(len(CLEAN)).all()
assert np.isfinite(CLEAN).all() and all(np.isfinite(x).all() for x in NOISY.values())
assert manifest.r.between(0.2, 1.99).all()
manifest.to_csv(OUT / 'manifest.csv', index=False)
display(manifest.groupby(['protocol', 'noise_channel']).size().rename('N').reset_index())


## 10. Chạy DDIM K = 1/3/5/10

In [ ]:
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def metric_rows(clean, noisy, pred, meta, protocol, k, batch_time_ms, batch_start):
    clean, noisy, pred = [np.asarray(a, dtype=np.float64) for a in (clean, noisy, pred)]
    diff = pred - clean
    numerator = np.sum(diff ** 2, axis=1)
    signal_power = np.sum(clean ** 2, axis=1)
    input_power = np.sum((noisy - clean) ** 2, axis=1)
    repo_denominator = np.sum((pred - np.mean(clean)) ** 2, axis=1)
    standard_denominator = np.sum((clean - clean.mean(axis=1, keepdims=True)) ** 2, axis=1)
    cosine_denominator = np.linalg.norm(clean, axis=1) * np.linalg.norm(pred, axis=1)
    def safe_ratio(num, den):
        return np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        snr_in = 10 * np.log10(safe_ratio(signal_power, input_power))
        snr_out = 10 * np.log10(safe_ratio(signal_power, numerator))
        values = {
            'ssd': numerator,
            'mad': np.max(np.abs(diff), axis=1),
            'prd_repo': 100 * np.sqrt(safe_ratio(numerator, repo_denominator)),
            'prd_standard': 100 * np.sqrt(safe_ratio(numerator, standard_denominator)),
            'cosine_similarity': safe_ratio(np.sum(clean * pred, axis=1), cosine_denominator),
            'snr_in_db': snr_in, 'snr_out_db': snr_out,
            'snr_improvement_db': snr_out - snr_in,
        }
    frame = meta[['sample_id', 'patient_id', 'record_id', 'noise_type', 'r']].copy()
    frame['experiment_id'] = 'qtdb_bw_rule_v1'
    frame['model_id'] = 'research_hnf_unet_ddpm_1ch'
    frame['checkpoint_id'] = CHECKPOINT_INFO[protocol - 1]['sha256']
    frame['train_seed'] = np.nan  # Plain state_dict does not prove training seed.
    frame['inference_seed'] = SEED + protocol
    frame['protocol'] = protocol
    frame['K'] = k
    frame['output_row'] = np.arange(batch_start, batch_start + len(clean))
    frame['output_file'] = f'xhat_protocol{protocol}_K{k}.npy'
    frame['inference_time_ms'] = batch_time_ms / len(clean)
    for key, vector in values.items():
        frame[key] = vector
    return frame

RESULT_FRAMES = []
for protocol in (1, 2):
    model = MODELS[protocol]
    predictions = {k: np.lib.format.open_memmap(
        OUT / f'xhat_protocol{protocol}_K{k}.npy', mode='w+',
        dtype=np.float32, shape=CLEAN.shape) for k in K_VALUES}
    torch.manual_seed(SEED + protocol)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(SEED + protocol)
    for start in tqdm(range(0, len(CLEAN), BATCH_SIZE), desc=f'Protocol {protocol}'):
        stop = min(start + BATCH_SIZE, len(CLEAN))
        condition = torch.from_numpy(NOISY[protocol][start:stop, None, :]).to(DEVICE)
        assert condition.shape == (stop - start, 1, LENGTH)
        running_sum = torch.zeros_like(condition)
        synchronize()
        tick = time.perf_counter()
        with torch.inference_mode():
            for shot in range(1, max(K_VALUES) + 1):
                output = model.denoising(condition, use_ddim=True,
                                         ddim_steps=DDIM_STEPS, ddim_eta=DDIM_ETA,
                                         num_shots=1)
                assert output.shape == condition.shape and torch.isfinite(output).all()
                running_sum += output
                if shot in K_VALUES:
                    synchronize()
                    elapsed_ms = (time.perf_counter() - tick) * 1000
                    average = (running_sum / shot).cpu().numpy()[:, 0, :]
                    predictions[shot][start:stop] = average
                    RESULT_FRAMES.append(metric_rows(
                        CLEAN[start:stop], NOISY[protocol][start:stop], average,
                        MANIFEST[protocol].iloc[start:stop].reset_index(drop=True),
                        protocol, shot, elapsed_ms, start))
    for array in predictions.values():
        array.flush()

per_sample = pd.concat(RESULT_FRAMES, ignore_index=True)
assert len(per_sample) == 2 * len(CLEAN) * len(K_VALUES)
per_sample.to_csv(OUT / 'metrics_per_sample.csv', index=False)
print('Per-sample rows:', len(per_sample))


## 11. Kiểm toán và tổng hợp metric

In [ ]:
METRICS = ['ssd', 'mad', 'prd_repo', 'prd_standard', 'cosine_similarity',
           'snr_in_db', 'snr_out_db', 'snr_improvement_db', 'inference_time_ms']
assert not per_sample.duplicated(['protocol', 'K', 'sample_id']).any()
assert np.allclose(per_sample.snr_improvement_db,
                   per_sample.snr_out_db - per_sample.snr_in_db, equal_nan=True)
per_sample['r_group'] = pd.cut(
    per_sample.r, bins=[0.2, 0.6, 1.0, 1.5, 2.001], right=False,
    labels=['0.20–<0.60', '0.60–<1.00', '1.00–<1.50', '1.50–2.00'])
assert per_sample.r_group.notna().all()
assert per_sample.groupby(['protocol', 'K'], observed=True).r_group.count().eq(len(CLEAN)).all()

def summarize(frame, protocol_label, group_label):
    result = {'model_id': 'research_hnf_unet_ddpm_1ch',
              'protocol': protocol_label, 'K': int(frame.K.iloc[0]),
              'r_group': group_label, 'N': len(frame)}
    for metric in METRICS:
        vector = frame[metric].to_numpy(dtype=np.float64)
        finite = vector[np.isfinite(vector)]
        result[f'{metric}_valid_N'] = len(finite)
        result[f'{metric}_nonfinite_N'] = len(vector) - len(finite)
        result[f'{metric}_mean'] = float(np.mean(finite)) if len(finite) else np.nan
        result[f'{metric}_std'] = float(np.std(finite, ddof=0)) if len(finite) else np.nan
    return result

summary_rows = []
for k in K_VALUES:
    subset_k = per_sample[per_sample.K == k]
    for protocol in (1, 2, 'pooled'):
        frame = subset_k if protocol == 'pooled' else subset_k[subset_k.protocol == protocol]
        summary_rows.append(summarize(frame, protocol, 'ALL'))
        for label, part in frame.groupby('r_group', observed=True):
            summary_rows.append(summarize(part, protocol, str(label)))
summary = pd.DataFrame(summary_rows)
assert summary[summary.r_group == 'ALL'].groupby('K').N.sum().eq(4 * len(CLEAN)).all()
for (protocol, k), frame in summary.groupby(['protocol', 'K']):
    assert frame.loc[frame.r_group != 'ALL', 'N'].sum() == frame.loc[frame.r_group == 'ALL', 'N'].iloc[0]
# Record means keep the independent QTDB record visible separately from segment-weighted tables.
record_rows = []
for (protocol, k, record_id), frame in per_sample.groupby(['protocol', 'K', 'record_id']):
    row = summarize(frame, protocol, 'ALL')
    row['record_id'] = record_id
    record_rows.append(row)
record_summary = pd.DataFrame(record_rows)
per_sample.to_csv(OUT / 'metrics_per_sample.csv', index=False)
summary.to_csv(OUT / 'summary.csv', index=False)
record_summary.to_csv(OUT / 'summary_by_record.csv', index=False)
readable = summary[['protocol', 'K', 'r_group', 'N']].copy()
for metric in METRICS:
    readable[metric] = [f'{mean:.4f} ± {std:.4f}'
                        for mean, std in zip(summary[f'{metric}_mean'], summary[f'{metric}_std'])]
readable.to_csv(OUT / 'summary_mean_std.csv', index=False)
display(readable.loc[readable.r_group == 'ALL'])
print('Grouped by r (segment-weighted):')
display(readable.loc[readable.r_group != 'ALL',
                     ['protocol', 'K', 'r_group', 'N', 'prd_repo', 'snr_improvement_db']])
display(summary.loc[summary.r_group == 'ALL',
                    ['protocol', 'K', 'N'] + [f'{m}_nonfinite_N' for m in METRICS]])


## 12. Biểu đồ và metadata

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for protocol in (1, 2):
    part = summary[(summary.protocol == protocol) & (summary.r_group == 'ALL')]
    axes[0].plot(part.K, part.snr_improvement_db_mean, marker='o', label=f'Protocol {protocol}')
    axes[1].plot(part.K, part.prd_repo_mean, marker='o', label=f'Protocol {protocol}')
axes[0].set(title='Cải thiện SNR theo K — QTDB/BW test', xlabel='K (số lần suy mẫu)', ylabel='ΔSNR (dB)')
axes[1].set(title='PRD_repo theo K — QTDB/BW test', xlabel='K (số lần suy mẫu)', ylabel='PRD_repo (%)')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'summary_by_K.png', dpi=160)
plt.show()

index = 0
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(CLEAN[index], label='QTDB clean', color='black')
ax.plot(NOISY[1][index], label='BW noisy (protocol 1)', alpha=0.65)
for k in (1, 10):
    estimate = np.load(OUT / f'xhat_protocol1_K{k}.npy', mmap_mode='r')[index]
    ax.plot(estimate, label=f'DDIM K={k}', alpha=0.85)
ax.set(xlabel='Mẫu tại 360 Hz', ylabel='mV', title=f'Mẫu {CLEAN_META.sample_id.iloc[index]} — protocol 1')
ax.legend(ncol=4)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / 'waveform_example.png', dpi=160)
plt.show()

run_info = {
    'source_clean': PHYSIONET_QT, 'source_noise': PHYSIONET_NST + '/bw',
    'test_records': list(selected_records), 'full_benchmark': MAX_TEST_RECORDS is None and MAX_TEST_SEGMENTS is None,
    'N_clean': len(CLEAN), 'N_per_protocol': len(CLEAN),
    'sampling_rate': TARGET_FS, 'segment_length': LENGTH,
    'preprocessing_version': PREPROCESSING_VERSION,
    'r_definition': 'peak-to-peak scaled BW / peak-to-peak clean; discrete 0.20..1.99',
    'noise_pair_seed': SEED, 'inference_seeds': {'1': SEED + 1, '2': SEED + 2},
    'K': list(K_VALUES), 'batch_size': BATCH_SIZE,
    'sampler': 'research DDPM.ddim_sample_loop', 'ddim_steps': DDIM_STEPS, 'eta': DDIM_ETA,
    'diffusion': CONFIG['diffusion'], 'model_base_features': 80,
    'model_embedding_dim': 128, 'checkpoint_info': CHECKPOINT_INFO,
    'training_configuration_from_research': CONFIG['train'],
    'declared_training_completed_epochs': DECLARED_TRAIN_EPOCHS,
    'checkpoint_selection': 'minimum validation loss during the completed 100-epoch training run',
    'device': str(DEVICE), 'torch_version': torch.__version__,
    'numpy_version': np.__version__, 'scipy_version': scipy.__version__,
    'wfdb_version': wfdb.__version__, 'python_version': platform.python_version(),
    'std_definition': 'population ddof=0 over finite per-segment values',
    'nonfinite_policy': 'keep all rows; aggregate finite metric values and report nonfinite_N per metric',
}
(OUT / 'run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', sorted(path.name for path in OUT.iterdir()))


## Output của Kaggle version

Sau khi Save & Run All hoàn tất, mở tab **Output** và tải thư mục `qtdb_rule_evaluation_100epoch`. Các tệp chính:

- `metrics_per_sample.csv`
- `summary.csv` và `summary_mean_std.csv`
- `summary_by_record.csv`
- `manifest.csv`
- `xhat_protocol*_K*.npy`
- `run_info.json`
- `summary_by_K.png` và `waveform_example.png`